# 🎭 L8 — Adapter le prompt au contexte avec `@dynamic_prompt`

> ⏱️ **Durée indicative : 30 à 45 minutes**  
> 🎓 **Niveau : débutant** — nous allons rendre chaque transformation visible avant d'appeler le modèle.

Un même assistant peut parler à plusieurs publics. Un visiteur a besoin du catalogue musical ; un employé peut aussi analyser les ventes. Au lieu de créer deux agents presque identiques, nous allons adapter leurs consignes au moment de chaque appel.

📚 Le mécanisme appartient au [context engineering de LangChain](https://docs.langchain.com/oss/python/langchain/context-engineering) et s'intègre grâce au [middleware](https://docs.langchain.com/oss/python/langchain/middleware/overview).

## 🎯 Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- expliquer la différence entre un prompt statique et un prompt dynamique ;
- transmettre un contexte d'exécution typé à un agent ;
- utiliser `@dynamic_prompt` pour produire des consignes différentes ;
- tester la transformation du prompt **sans appeler de modèle** ;
- distinguer personnalisation du prompt, lecture seule et véritable contrôle d'accès.

### 🧭 Signalétique

🎯 objectif · 🧠 intuition · 🗺️ schéma mental · 🛠️ construction · 🔮 prédiction · ▶️ exécution · 👀 observation · 🔍 vérification · ⚠️ piège · 🧪 exercice · ✅ correction · 📚 documentation

## 🗺️ La carte mentale

```text
Question + RuntimeContext(role=...)
                  │
                  ▼
        🎛️ middleware @dynamic_prompt
                  │ construit le prompt adapté
                  ▼
        🧠 Mistral choisit sa réponse / un outil
                  │
                  ▼
        🐍 LangChain exécute éventuellement l'outil SQL
```

🧠 **Analogie ELI5 adulte :** le contexte est le badge porté par la personne. Le middleware est l'accueil qui lit le badge et remet la bonne fiche de consignes à l'assistant. Le badge ne modifie pas le modèle ; il modifie ce que le modèle lit pour cette requête.

## 🔐 1. Configurer Mistral sans lire de fichier `.env`

Le notebook consomme uniquement `MISTRAL_API_KEY` et `MISTRAL_SERVER_URL` déjà présentes dans le processus Jupyter. Il ne lit ni n'affiche aucun secret.

L'endpoint est normalisé pour accepter une URL avec ou sans suffixe `/v1`.

📚 Consultez l'[intégration officielle `ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai) pour les paramètres du modèle.

In [1]:
import os

from langchain_mistralai import ChatMistralAI


def normalize_mistral_endpoint(raw_url: str) -> str:
    """Return the API base URL expected by ChatMistralAI."""
    endpoint = raw_url.strip().rstrip("/")
    endpoint = endpoint.removesuffix("/chat/completions")
    if not endpoint.endswith("/v1"):
        endpoint = f"{endpoint}/v1"
    return endpoint


required_variables = ("MISTRAL_API_KEY", "MISTRAL_SERVER_URL")
missing_variables = [name for name in required_variables if not os.environ.get(name)]
if missing_variables:
    raise RuntimeError(
        "❌ Variables manquantes dans le processus Jupyter : "
        + ", ".join(missing_variables)
        + ". Définissez-les avant de lancer Jupyter, puis redémarrez le kernel."
    )

mistral_endpoint = normalize_mistral_endpoint(os.environ["MISTRAL_SERVER_URL"])

# 📚 API officielle : https://docs.langchain.com/oss/python/integrations/chat/mistralai
mistral_model = ChatMistralAI(
    model="mistral-medium-latest",
    temperature=0,
    api_key=os.environ["MISTRAL_API_KEY"],
    endpoint=mistral_endpoint,
)

print("✅ Configuration Mistral détectée — aucune valeur n'est affichée.")

✅ Configuration Mistral détectée — aucune valeur n'est affichée.


## 🗄️ 2. Ouvrir Chinook réellement en lecture seule

Notre assistant analysera la base d'exemple Chinook. Deux précautions évitent une démonstration trompeuse :

1. SQLite est ouvert avec `mode=ro` : une écriture sera refusée par la base elle-même ;
2. nous donnerons au modèle un schéma explicite : il n'aura pas à inventer les tables ou colonnes.

⚠️ **Lecture seule ne signifie pas contrôle d'accès.** Elle empêche les modifications, mais n'empêche pas de lire une table sensible. Nous reviendrons sur ce point.

In [2]:
from pathlib import Path

from sql_db import SQLDatabase

database_path = Path("Chinook.db")
if not database_path.exists():
    raise FileNotFoundError(
        "❌ Chinook.db est introuvable. Lancez Jupyter depuis le dossier 'J2_Hands On'."
    )

# `uri=true` demande à SQLite d'interpréter `mode=ro` : la protection
# lecture seule est appliquée par le moteur, pas seulement par le prompt.
db = SQLDatabase.from_uri("sqlite:///file:Chinook.db?mode=ro&uri=true")

print("✅ Base ouverte en lecture seule.")
print("Tables disponibles :", ", ".join(db.get_usable_table_names()))

✅ Base ouverte en lecture seule.
Tables disponibles : Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track


## 📐 3. Donner un plan exact de la base au modèle

🧠 Demander du SQL sans fournir de schéma revient à demander un itinéraire sans carte. Le modèle peut connaître la syntaxe SQL, mais il ne peut pas deviner de façon fiable nos noms de colonnes.

Nous séparons :

- le **catalogue public**, utile à un visiteur ;
- les **données internes**, nécessaires pour analyser clients, employés et ventes.

In [3]:
PUBLIC_SCHEMA = """
Artist(ArtistId, Name)
Album(AlbumId, Title, ArtistId)
Track(TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice)
Genre(GenreId, Name)
MediaType(MediaTypeId, Name)
Playlist(PlaylistId, Name)
PlaylistTrack(PlaylistId, TrackId)
""".strip()

INTERNAL_SCHEMA = """
Customer(CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId)
Employee(EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email)
Invoice(InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total)
InvoiceLine(InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)
""".strip()

FULL_SCHEMA = f"{PUBLIC_SCHEMA}\n{INTERNAL_SCHEMA}"
print(FULL_SCHEMA)

Artist(ArtistId, Name)
Album(AlbumId, Title, ArtistId)
Track(TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice)
Genre(GenreId, Name)
MediaType(MediaTypeId, Name)
Playlist(PlaylistId, Name)
PlaylistTrack(PlaylistId, TrackId)
Customer(CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId)
Employee(EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email)
Invoice(InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total)
InvoiceLine(InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)


### 👀 Ce qu'il faut observer

- Les relations utiles deviennent visibles : `Invoice.CustomerId → Customer.CustomerId`, par exemple.
- Le modèle n'aura pas besoin d'essayer des colonnes imaginaires comme `Customer.FullName`.
- Nous contrôlons exactement les informations de schéma insérées dans chaque prompt.

## 🧱 4. Premier essai : un prompt statique

Un **prompt statique** est fixé à la création de l'agent. Il convient lorsque les mêmes consignes s'appliquent toujours.

### 🔮 Pause prédiction

Si nous mettons tout le schéma dans ce prompt, que recevra un visiteur ? Et un employé ?

➡️ Les deux recevront exactement la même chose : le prompt ne connaît pas leur contexte.

In [4]:
STATIC_SYSTEM_PROMPT = f"""Tu es un analyste SQLite prudent.
Utilise l'outil SQL pour répondre à partir du schéma suivant :

{FULL_SCHEMA}

N'exécute qu'une requête SELECT à la fois et limite les résultats à 5 lignes.
"""

static_prompt_for_visitor = STATIC_SYSTEM_PROMPT
static_prompt_for_employee = STATIC_SYSTEM_PROMPT

print("Prompts identiques :", static_prompt_for_visitor == static_prompt_for_employee)
print("Le visiteur voit Invoice :", "Invoice(" in static_prompt_for_visitor)

Prompts identiques : True
Le visiteur voit Invoice : True


### 👀 Observation

Le test affiche `True` deux fois. Le prompt statique fonctionne techniquement, mais il ne peut pas adapter son vocabulaire ou son périmètre au rôle courant.

🧠 Nous pourrions créer deux agents, mais nous dupliquerions leur modèle, leurs outils et leurs règles communes. Utilisons plutôt un contexte d'exécution.

## 🪪 5. Définir le contexte d'exécution

Le **runtime context** transporte des informations disponibles pendant un appel, sans les ajouter comme un message utilisateur. Ici il contient :

- `role`, égal à `visitor` ou `employee` ;
- la connexion `db`, utilisée par l'outil SQL.

📚 LangChain présente ce contexte dans sa documentation sur le [context engineering](https://docs.langchain.com/oss/python/langchain/context-engineering).

In [5]:
from dataclasses import dataclass
from typing import Literal


@dataclass
class RuntimeContext:
    """Information available only for the duration of one agent run."""

    role: Literal["visitor", "employee"]
    db: SQLDatabase


visitor_context = RuntimeContext(role="visitor", db=db)
employee_context = RuntimeContext(role="employee", db=db)
print(visitor_context.role, "/", employee_context.role)

visitor / employee


## 🛠️ 6. Construire une transformation pure et testable

Avant le décorateur LangChain, écrivons une fonction Python ordinaire. Elle reçoit un rôle et retourne le prompt correspondant.

Cette séparation est importante :

- la logique devient facile à lire ;
- nous pouvons la tester sans réseau et sans coût ;
- le middleware restera une très fine couche d'intégration.

In [6]:
def build_system_prompt(role: Literal["visitor", "employee"]) -> str:
    """Build the role-aware SQL instructions used by the middleware."""
    common_rules = """Règles communes :
- Utilise execute_readonly_sql seulement si les données sont nécessaires.
- Produis UNE requête SQLite SELECT (ou WITH suivi d'un SELECT) à la fois.
- Utilise uniquement les tables et colonnes du schéma fourni.
- Préfère des colonnes explicites à SELECT *.
- Ajoute LIMIT 5 sauf si la question demande un agrégat unique.
- Si une information n'est pas disponible dans le schéma autorisé, dis-le clairement.
- Ne fabrique jamais une donnée absente du résultat SQL."""

    if role == "visitor":
        role_rules = (
            "Tu aides un visiteur à explorer le catalogue musical public. "
            "Refuse poliment les questions qui nécessitent des données clients, "
            "employés ou factures."
        )
        allowed_schema = PUBLIC_SCHEMA
    elif role == "employee":
        role_rules = (
            "Tu aides un employé autorisé à analyser le catalogue et les ventes. "
            "Tu peux utiliser l'ensemble du schéma fourni."
        )
        allowed_schema = FULL_SCHEMA
    else:
        raise ValueError(f"Rôle inconnu : {role}")

    return f"""Tu es un analyste SQLite prudent.

Contexte du rôle :
{role_rules}

Schéma autorisé :
{allowed_schema}

{common_rules}
"""

### 🔮 Pause prédiction

Avant le test, prédisez :

- le prompt visiteur contiendra-t-il `Invoice` ?
- le prompt employé contiendra-t-il `Invoice` ?
- les règles SQL communes seront-elles présentes dans les deux ?

In [7]:
from difflib import unified_diff

visitor_prompt = build_system_prompt("visitor")
employee_prompt = build_system_prompt("employee")

print("=== PROMPT VISITEUR ===")
print(visitor_prompt)
print("\n=== DIFF VISITEUR → EMPLOYÉ ===")
print(
    "".join(
        unified_diff(
            visitor_prompt.splitlines(keepends=True),
            employee_prompt.splitlines(keepends=True),
            fromfile="visitor",
            tofile="employee",
        )
    )
)

assert "Invoice(" not in visitor_prompt
assert "Invoice(" in employee_prompt
assert "Ne fabrique jamais" in visitor_prompt and "Ne fabrique jamais" in employee_prompt
print("✅ Les deux variantes respectent le contrat attendu.")

=== PROMPT VISITEUR ===
Tu es un analyste SQLite prudent.

Contexte du rôle :
Tu aides un visiteur à explorer le catalogue musical public. Refuse poliment les questions qui nécessitent des données clients, employés ou factures.

Schéma autorisé :
Artist(ArtistId, Name)
Album(AlbumId, Title, ArtistId)
Track(TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice)
Genre(GenreId, Name)
MediaType(MediaTypeId, Name)
Playlist(PlaylistId, Name)
PlaylistTrack(PlaylistId, TrackId)

Règles communes :
- Utilise execute_readonly_sql seulement si les données sont nécessaires.
- Produis UNE requête SQLite SELECT (ou WITH suivi d'un SELECT) à la fois.
- Utilise uniquement les tables et colonnes du schéma fourni.
- Préfère des colonnes explicites à SELECT *.
- Ajoute LIMIT 5 sauf si la question demande un agrégat unique.
- Si une information n'est pas disponible dans le schéma autorisé, dis-le clairement.
- Ne fabrique jamais une donnée absente du résultat SQL.


=== DIF

### 👀 Observation

Le diff rend la transformation concrète : le rôle, le périmètre et le schéma changent ; les règles communes restent stables.

✅ Nous venons de tester le comportement essentiel **sans appeler Mistral**. Si ce test échoue, le problème se trouve dans notre logique Python, pas dans le modèle.

## 🎛️ 7. Brancher la fonction au middleware `@dynamic_prompt`

Un **middleware** est une étape qui s'insère dans le cycle de l'agent. `@dynamic_prompt` exécute notre fonction avant l'appel au modèle et remplace le prompt système pour cet appel.

Le `ModelRequest` donne accès au runtime courant. Nous lisons `request.runtime.context.role`, puis déléguons toute la logique à notre fonction testée.

📚 Références : [middleware LangChain](https://docs.langchain.com/oss/python/langchain/middleware/overview) et [context engineering](https://docs.langchain.com/oss/python/langchain/context-engineering).

In [8]:
from langchain.agents.middleware import ModelRequest, dynamic_prompt


# 📚 @dynamic_prompt et ModelRequest :
# https://docs.langchain.com/oss/python/langchain/context-engineering
@dynamic_prompt
def role_aware_system_prompt(request: ModelRequest) -> str:
    """Select the tested prompt variant from the current runtime context."""
    return build_system_prompt(request.runtime.context.role)

## 🧰 8. Créer l'outil SQL en lecture seule

L'outil récupère la connexion depuis le même runtime context. Il accepte seulement une instruction commençant par `SELECT` ou `WITH`, puis SQLite applique sa propre protection `mode=ro`.

📚 Le décorateur [`@tool`](https://docs.langchain.com/oss/python/langchain/tools) transforme la fonction et sa docstring en contrat lisible par l'agent.

In [9]:
from langchain.tools import tool
from langgraph.runtime import get_runtime


# 📚 Tools : https://docs.langchain.com/oss/python/langchain/tools
# 📚 Runtime context : https://docs.langchain.com/oss/python/langchain/context-engineering
@tool
def execute_readonly_sql(query: str) -> str:
    """Execute one read-only SQLite SELECT query and return at most its result."""
    runtime = get_runtime(RuntimeContext)
    runtime_db = runtime.context.db

    cleaned_query = query.strip()
    first_keyword = cleaned_query.split(maxsplit=1)[0].upper() if cleaned_query else ""
    if first_keyword not in {"SELECT", "WITH"}:
        return "Error: seules les requêtes SELECT en lecture sont autorisées."
    if ";" in cleaned_query.rstrip(";"):
        return "Error: une seule instruction SQL est autorisée."

    try:
        return runtime_db.run(cleaned_query)
    except Exception as error:
        return f"Error: {error}"

## 🤖 9. Assembler un seul agent, deux comportements

[`create_agent`](https://docs.langchain.com/oss/python/langchain/agents) reçoit :

- le même modèle Mistral ;
- le même outil SQL ;
- notre middleware dynamique ;
- le schéma du contexte pour valider `role` et `db`.

Il n'y a donc qu'un seul agent. C'est le contexte fourni à chaque invocation qui change la fiche de consignes.

In [10]:
from langchain.agents import create_agent

# 📚 create_agent : https://docs.langchain.com/oss/python/langchain/agents
agent = create_agent(
    model=mistral_model,
    tools=[execute_readonly_sql],
    middleware=[role_aware_system_prompt],
    context_schema=RuntimeContext,
)

print("✅ Agent unique créé avec un prompt dynamique.")

✅ Agent unique créé avec un prompt dynamique.


## 👤 10. Même question, contexte visiteur

Nous reprenons le scénario du cours : rechercher l'achat le plus coûteux de Frank Harris. Cette question exige les tables internes `Customer` et `Invoice`.

### 🔮 Pause prédiction

Avec `role="visitor"`, le prompt dynamique contient-il ces tables ? L'agent devrait-il exécuter l'outil ou expliquer sa limite ?

In [11]:
question = "Quel est l'achat le plus coûteux de Frank Harris ?"

visitor_result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    context=visitor_context,
)

for message in visitor_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Quel est l'achat le plus coûteux de Frank Harris ?
================================== Ai Message ==================================

Je ne peux pas répondre à cette question, car elle implique des données clients ou des informations sur les achats, qui ne font pas partie du schéma autorisé. Le schéma autorisé ne contient que des informations sur les artistes, les albums, les pistes, les genres, les types de médias, les playlists et les pistes des playlists.

Si vous avez une question concernant ces données, je serai ravi de vous aider !


### 👀 Résultat attendu

L'agent devrait expliquer qu'il ne dispose que du catalogue public et ne peut pas répondre à une question sur les achats d'un client. Aucun `ToolMessage` ne devrait apparaître.

🔍 Si un appel SQL apparaît malgré tout, ce n'est pas une autorisation valable : c'est précisément la preuve qu'une consigne en langage naturel peut être ignorée.

## 🧑‍💼 11. Même question, contexte employé

### 🔮 Pause prédiction

Avec `role="employee"`, quelles tables devraient être jointes ?

Une stratégie possible est : `Customer → Invoice`, tri décroissant sur `Invoice.Total`, puis `LIMIT 1`.

In [12]:
employee_result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    context=employee_context,
)

for message in employee_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Quel est l'achat le plus coûteux de Frank Harris ?
================================== Ai Message ==================================
Tool Calls:
  execute_readonly_sql (y3PcI9LoU)
 Call ID: y3PcI9LoU
  Args:
    query: SELECT Invoice.Total AS MontantTotal, Invoice.InvoiceId, Invoice.InvoiceDate
FROM Invoice
JOIN Customer ON Invoice.CustomerId = Customer.CustomerId
WHERE Customer.FirstName = 'Frank' AND Customer.LastName = 'Harris'
ORDER BY Invoice.Total DESC
LIMIT 1;
================================= Tool Message =================================
Name: execute_readonly_sql

[(13.86, 145, '2010-09-23 00:00:00')]
================================== Ai Message ==================================

L'achat le plus coûteux de **Frank Harris** est une facture d'un montant total de **13,86 $**, émise le **23 septembre 2010** (ID de facture : **145**).


### 🔍 Anatomie de la seconde exécution

| Étape | Responsable |
|---|---|
| Lit `role=employee` | runtime context |
| Construit le prompt complet | middleware `@dynamic_prompt` |
| Choisit l'outil et génère le SQL | Mistral |
| Exécute l'appel d'outil | LangChain/Python |
| Refuse physiquement une écriture | SQLite en `mode=ro` |
| Formule la réponse finale | Mistral |

Le modèle n'a ni changé ni appris entre les deux appels. Seul le contexte runtime a produit un prompt différent.

## 🚨 Sécurité : un prompt n'est **pas** un contrôle d'accès

> ⚠️ **Ne reproduisez jamais cette démonstration comme mécanisme d'autorisation en production.**

Le prompt visiteur demande au modèle de ne pas consulter certaines tables, mais l'outil SQL possède techniquement la même connexion dans les deux contextes. Un modèle peut se tromper, être manipulé par une injection de prompt ou générer une requête inattendue.

En production, appliquez les droits **hors du modèle**, par exemple avec :

- des comptes de base distincts et des permissions SQL minimales ;
- des vues dédiées aux données publiques ;
- une allowlist vérifiée côté application ;
- un outil différent selon le rôle ;
- des journaux d'audit et des tests d'injection.

✅ Ici, `mode=ro` protège bien contre les écritures. Il ne protège **pas** la confidentialité des lectures. `@dynamic_prompt` personnalise le comportement ; il n'accorde ni ne retire un droit.

## 🧪 Micro-exercice — Auditer le prompt sans appeler Mistral

Complétez la cellule suivante pour vérifier automatiquement qu'un visiteur voit `Track`, mais ne voit ni `Customer` ni `Invoice`.

### ✅ Critères de réussite

- vous construisez le prompt avec `build_system_prompt("visitor")` ;
- trois assertions vérifient les tables demandées ;
- la cellule affiche le message de succès ;
- aucun appel au modèle ou à la base n'est effectué.

In [13]:
# TODO 🧪 Décommentez puis complétez les assertions.
# prompt_to_audit = build_system_prompt("...")
# assert "Track(" ... prompt_to_audit
# assert "Customer(" ... prompt_to_audit
# assert "Invoice(" ... prompt_to_audit
# print("✅ Le périmètre visiteur est correctement décrit dans le prompt.")

<details>
<summary>✅ Afficher la correction</summary>

```python
prompt_to_audit = build_system_prompt("visitor")
assert "Track(" in prompt_to_audit
assert "Customer(" not in prompt_to_audit
assert "Invoice(" not in prompt_to_audit
print("✅ Le périmètre visiteur est correctement décrit dans le prompt.")
```

🔍 Ce test vérifie notre transformation déterministe. Il ne prouve pas que le modèle respectera toujours le texte : les autorisations doivent rester côté application et base de données.
</details>

## ⚠️ Pièges fréquents

- **Mettre le rôle dans le message utilisateur** : l'utilisateur pourrait le modifier. Le contexte runtime doit venir de votre application après authentification.
- **Tester seulement avec le modèle** : testez d'abord `build_system_prompt`, plus rapide et déterministe.
- **Laisser le modèle inventer le schéma** : fournissez les tables et colonnes exactes.
- **Confondre `mode=ro` et confidentialité** : la lecture seule bloque les écritures, pas les lectures sensibles.
- **Utiliser un prompt comme ACL** : appliquez les permissions dans les outils, les vues et la base.
- **Oublier `context_schema`** : vous perdez la validation et rendez le contrat de l'agent moins lisible.

## ✅ Ce que vous savez maintenant

Vous savez désormais que :

1. un prompt statique est identique pour tous les appels ;
2. le runtime context transporte des informations propres à une exécution ;
3. `@dynamic_prompt` est un middleware qui construit le prompt juste avant Mistral ;
4. une fonction pure rend la transformation directement observable et testable ;
5. un schéma explicite réduit les hallucinations SQL ;
6. un prompt dynamique n'est jamais une autorisation de sécurité.

## 🧭 Transition vers L9

Nous savons maintenant adapter les consignes automatiquement. Dans **L9 — Human-in-the-Loop**, nous irons plus loin : l'agent s'interrompra avant une action et attendra une vraie décision humaine — approuver ou rejeter.

## 📚 Documentation officielle

- [LangChain — Context engineering](https://docs.langchain.com/oss/python/langchain/context-engineering)
- [LangChain — Vue d'ensemble du middleware](https://docs.langchain.com/oss/python/langchain/middleware/overview)
- [LangChain — Agents et `create_agent`](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain — Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain — Intégration `ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai)
- [Mistral AI — Function calling](https://docs.mistral.ai/studio/conversations/function-calling)
- [SQLite — URI filenames et paramètre `mode`](https://www.sqlite.org/uri.html)